In [4]:
import glob
import numpy as np
import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

Using: cuda


In [9]:
from google.colab import drive
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/sudoko"
files = glob.glob(SAVE_DIR + "/*.npz")
for f in files:
    print(f)

def load_file(name):
    for f in files:
        if f.endswith("/" + name + ".npz"):
            return np.load(f)

val = load_file("val")
train = load_file("train_n1000_k1")

print("train:", train["puzzles"].shape)
print("val:", val["puzzles"].shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/sudoko/train_n100_k100.npz
/content/drive/MyDrive/sudoko/train_n1000_k100.npz
/content/drive/MyDrive/sudoko/train_n10000_k1.npz
/content/drive/MyDrive/sudoko/train_n1000_k10.npz
/content/drive/MyDrive/sudoko/train_n1000_k1.npz
/content/drive/MyDrive/sudoko/train_n10_k1000.npz
/content/drive/MyDrive/sudoko/train_n100_k100 (1).npz
/content/drive/MyDrive/sudoko/val.npz
/content/drive/MyDrive/sudoko/test.npz
/content/drive/MyDrive/sudoko/train_n100_k1.npz
/content/drive/MyDrive/sudoko/pool.npz
train: (1000, 81)
val: (10000, 81)


In [10]:
# Why: PyTorch models only work with tensors
# solutions - 1: the model's answers are numbered 0-8, so digit 1 -> 0, digit 9 -> 8
def to_tensors(data):
    puzzles = torch.tensor(data["puzzles"], dtype=torch.long)
    solutions = torch.tensor(data["solutions"], dtype=torch.long) - 1
    return puzzles, solutions

train_p, train_s = to_tensors(train)

print(train_p[0][:9])
print(train_s[0][:9])

tensor([1, 0, 0, 0, 3, 0, 0, 5, 0])
tensor([0, 7, 1, 8, 2, 6, 3, 4, 5])


In [11]:
class SudokuModel(nn.Module):
    def __init__(self, size=128, layers=4, heads=4):
        super().__init__()
        self.digit_embedding = nn.Embedding(10, size)
        self.position_embedding = nn.Embedding(81, size)
        one_layer = nn.TransformerEncoderLayer(
            d_model=size,
            nhead=heads,
            dim_feedforward=size * 4,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(one_layer, num_layers=layers)
        self.output = nn.Linear(size, 9)

    def forward(self, puzzles):
        positions = torch.arange(81, device=puzzles.device)
        x = self.digit_embedding(puzzles) + self.position_embedding(positions)
        x = self.transformer(x)
        scores = self.output(x)
        return scores

In [12]:
model = SudokuModel().to(device)

n_params = 0
for p in model.parameters():
    n_params += p.numel()
print("Parameters:", n_params)

scores = model(train_p[:2].to(device))
print(scores.shape)

Parameters: 805897
torch.Size([2, 81, 9])


In [13]:
def train_model(model, puzzles, solutions, steps, batch_size=64, lr=0.0005):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    model.train()

    for step in range(steps):
        # 1. pick random puzzles
        idx = torch.randint(0, len(puzzles), (batch_size,))
        x = puzzles[idx].to(device)
        y = solutions[idx].to(device)

        # 2. the model guesses, and we measure how wrong it is
        scores = model(x)
        loss = loss_fn(scores.reshape(-1, 9), y.reshape(-1))

        # 3 + 4. learn from the mistake
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 200 == 0:
            print("step", step, "loss", round(loss.item(), 4))

In [14]:
def evaluate(model, puzzles, solutions, batch_size=500):
    model.eval()
    empty_correct = 0
    empty_total = 0
    solved = 0

    with torch.no_grad():
        for start in range(0, len(puzzles), batch_size):
            x = puzzles[start:start + batch_size].to(device)
            y = solutions[start:start + batch_size].to(device)

            prediction = model(x).argmax(dim=-1)
            correct = prediction == y
            empty = x == 0

            empty_correct += (correct & empty).sum().item()
            empty_total += empty.sum().item()
            solved += correct.all(dim=1).sum().item()

    cell_acc = empty_correct / empty_total
    puzzle_acc = solved / len(puzzles)
    return cell_acc, puzzle_acc

In [15]:
small_p = train_p[:10]
small_s = train_s[:10]

model = SudokuModel().to(device)
train_model(model, small_p, small_s, steps=1000, batch_size=10)

cell_acc, puzzle_acc = evaluate(model, small_p, small_s)
print("ON THE 10 TRAINING PUZZLES")
print("Empty cells correct:", round(cell_acc * 100, 1), "%")
print("Puzzles solved:", round(puzzle_acc * 100, 1), "%")

step 0 loss 2.5505
step 200 loss 0.0107
step 400 loss 0.0027
step 600 loss 0.0037
step 800 loss 0.0066
ON THE 10 TRAINING PUZZLES
Empty cells correct: 100.0 %
Puzzles solved: 100.0 %


In [16]:
val_p, val_s = to_tensors(val)

cell_acc, puzzle_acc = evaluate(model, val_p, val_s)
print("ON NEW PUZZLES (validation)")
print("Empty cells correct:", round(cell_acc * 100, 1), "%")
print("Puzzles solved:", round(puzzle_acc * 100, 1), "%")

ON NEW PUZZLES (validation)
Empty cells correct: 11.4 %
Puzzles solved: 0.0 %


In [17]:
def train_with_checks(model, train_p, train_s, val_p, val_s, steps, check_every=1000, batch_size=64, lr=0.0005):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    model.train()

    for step in range(steps):
        # pick random puzzles, guess, measure, learn
        idx = torch.randint(0, len(train_p), (batch_size,))
        x = train_p[idx].to(device)
        y = train_s[idx].to(device)

        scores = model(x)
        loss = loss_fn(scores.reshape(-1, 9), y.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # every check_every steps: check training and validation scores
        if (step + 1) % check_every == 0:
            train_cell, train_puzzle = evaluate(model, train_p[:1000], train_s[:1000])
            val_cell, val_puzzle = evaluate(model, val_p, val_s)
            print("step", step + 1,
                  "| loss", round(loss.item(), 3),
                  "| TRAIN cells", round(train_cell * 100, 1), "% solved", round(train_puzzle * 100, 1), "%",
                  "| VAL cells", round(val_cell * 100, 1), "% solved", round(val_puzzle * 100, 1), "%")
            model.train()

In [18]:
train = load_file("train_n10000_k1")
train_p, train_s = to_tensors(train)
val_p, val_s = to_tensors(val)

model = SudokuModel().to(device)
train_with_checks(model, train_p, train_s, val_p, val_s, steps=10000)

step 1000 | loss 0.896 | TRAIN cells 47.5 % solved 0.0 % | VAL cells 46.3 % solved 0.0 %
step 2000 | loss 0.818 | TRAIN cells 51.9 % solved 0.0 % | VAL cells 49.7 % solved 0.0 %
step 3000 | loss 0.772 | TRAIN cells 54.6 % solved 0.0 % | VAL cells 51.0 % solved 0.0 %
step 4000 | loss 0.747 | TRAIN cells 57.7 % solved 0.0 % | VAL cells 52.1 % solved 0.0 %
step 5000 | loss 0.726 | TRAIN cells 59.9 % solved 0.0 % | VAL cells 52.6 % solved 0.0 %
step 6000 | loss 0.709 | TRAIN cells 62.1 % solved 0.0 % | VAL cells 52.9 % solved 0.0 %
step 7000 | loss 0.676 | TRAIN cells 63.9 % solved 0.0 % | VAL cells 53.2 % solved 0.0 %
step 8000 | loss 0.674 | TRAIN cells 65.6 % solved 0.0 % | VAL cells 53.3 % solved 0.0 %
step 9000 | loss 0.672 | TRAIN cells 67.2 % solved 0.0 % | VAL cells 53.3 % solved 0.0 %
step 10000 | loss 0.642 | TRAIN cells 68.9 % solved 0.0 % | VAL cells 53.4 % solved 0.0 %


In [19]:
def evaluate_by_group(model, data):
    puzzles, solutions = to_tensors(data)
    groups = data["groups"]
    for g in ["easy", "medium", "hard"]:
        mask = torch.tensor(groups == g)
        cell_acc, puzzle_acc = evaluate(model, puzzles[mask], solutions[mask])
        print(g, "- empty cells:", round(cell_acc * 100, 1), "%  solved:", round(puzzle_acc * 100, 1), "%")

evaluate_by_group(model, val)

easy - empty cells: 54.5 %  solved: 0.0 %
medium - empty cells: 53.2 %  solved: 0.0 %
hard - empty cells: 51.7 %  solved: 0.0 %


In [20]:
pool = load_file("pool")
pool_p, pool_s = to_tensors(pool)
print(pool_p.shape)

model_big = SudokuModel().to(device)
train_with_checks(model_big, pool_p, pool_s, val_p, val_s, steps=10000)

torch.Size([98000, 81])
step 1000 | loss 0.907 | TRAIN cells 46.3 % solved 0.0 % | VAL cells 46.3 % solved 0.0 %
step 2000 | loss 0.825 | TRAIN cells 49.6 % solved 0.0 % | VAL cells 49.2 % solved 0.0 %
step 3000 | loss 0.799 | TRAIN cells 50.1 % solved 0.0 % | VAL cells 49.9 % solved 0.0 %
step 4000 | loss 0.802 | TRAIN cells 50.7 % solved 0.0 % | VAL cells 50.4 % solved 0.0 %
step 5000 | loss 0.8 | TRAIN cells 51.3 % solved 0.0 % | VAL cells 51.0 % solved 0.0 %
step 6000 | loss 0.774 | TRAIN cells 51.8 % solved 0.0 % | VAL cells 51.6 % solved 0.0 %
step 7000 | loss 0.77 | TRAIN cells 53.0 % solved 0.0 % | VAL cells 52.8 % solved 0.0 %
step 8000 | loss 0.762 | TRAIN cells 54.9 % solved 0.0 % | VAL cells 54.6 % solved 0.0 %
step 9000 | loss 0.735 | TRAIN cells 56.7 % solved 0.0 % | VAL cells 56.6 % solved 0.0 %
step 10000 | loss 0.711 | TRAIN cells 57.6 % solved 0.0 % | VAL cells 57.5 % solved 0.0 %
